# SBD Model Demo (Запрос 10)

- **BaseEntity**: обработчики по `action` (минимум шаблонного кода в сущностях).
- **SimpleBroker**: маршрутизация сообщений.
- **YAML** (`demo_state.yaml`): снимок данных всех систем между сценариями.
- **Сценарий 1**: сертификация прошивки, регистрация БАС, закупка и приписка к `DronePort`.
- **Сценарий 2**: заказ заказчика на агро-обработку поля (e2e до посадки) на уже накопленном состоянии.

## Подключение кода демо

Каталог `notebooks/sbd-model-demo-code` должен быть в `sys.path`.

In [ ]:
import sys
from pathlib import Path

demo_code = Path.cwd() / "notebooks" / "sbd-model-demo-code"
if str(demo_code.resolve()) not in sys.path:
    sys.path.insert(0, str(demo_code.resolve()))

from sbd_demo import (
    default_state_path,
    reset_context,
    scenario_certification_and_purchase,
    scenario_customer_agro_order,
)
from world_state import load_world_state

STATE_YAML = default_state_path()
STATE_YAML

## Хранилище YAML и сброс контекста

Функция `reset_context(path)` перезаписывает файл дефолтным миром (как «очистка БД»). Перед чистым прогоном сценариев вызовите её явно.

In [ ]:
world_fresh = reset_context(STATE_YAML)
assert world_fresh["uas_registry"] == {}
len(load_world_state(STATE_YAML)["dronports"]["droneport_B"]["uas"])

## Сценарий 1: сертификация и закупка БАС

```mermaid
flowchart LR
  orchestrator --> vendor_uas
  vendor_uas --> regulator
  operator_1 --> vendor_uas
  operator_1 --> droneport_B
```

После выполнения состояние мержится в YAML (реестр UAS, парк дронопорта, список сертификатов).

In [ ]:
onboarding = scenario_certification_and_purchase(yaml_path=STATE_YAML)
onboarding["firmware_cert"]["certificate_id"], onboarding["purchase"]["registered_uas_ids"]

## Сценарий 2: заказ заказчика (агро-поле)

Используется мир, загруженный из YAML (включая уже купленные БАС на `droneport_B`). Полный цикл: Customer → Aggregator → Operator → GCS → ATM → AgroDrone → посадка.

In [ ]:
agro = scenario_customer_agro_order(yaml_path=STATE_YAML)
final = agro["final_order_result"]
assert final["status"] == "ok"
final["order_execution_completed"]["landing_coordinates"], final["selected_operator"]

## Просмотр сохранённого состояния (опционально)

In [ ]:
snap = load_world_state(STATE_YAML)
{
    "n_certs": len(snap.get("firmware_certificates", [])),
    "registry_size": len(snap.get("uas_registry", {})),
    "droneport_B_uas_count": len(snap["dronports"]["droneport_B"]["uas"]),
}